In [0]:

# ============================================================
# AUTOMATIC FILE REGISTRATION
# Cell 1 - Notebook parameters
# ============================================================

dbutils.widgets.text("storage_account", "sthealthcarede2026")
dbutils.widgets.text("source_container", "landing")
dbutils.widgets.text("source_folder", "patients")
dbutils.widgets.text("discovered_files_json", "[]")
dbutils.widgets.text("adf_run_id", "")

storage_account = dbutils.widgets.get("storage_account").strip()
source_container = dbutils.widgets.get("source_container").strip()
source_folder = dbutils.widgets.get("source_folder").strip("/")
discovered_files_json = dbutils.widgets.get("discovered_files_json")
adf_run_id = dbutils.widgets.get("adf_run_id").strip()

control_table = "dbw_healthcare_de_dev.etl.file_control"

print("Storage account:", storage_account)
print("Source container:", source_container)
print("Source folder:", source_folder)
print("ADF Run ID:", adf_run_id)


In [0]:

# ============================================================
# AUTOMATIC FILE REGISTRATION
# Cell 2 - Parse and validate discovered files
# ============================================================

import json

# Parse JSON received from ADF Get Metadata
try:
    discovered_files = json.loads(discovered_files_json)
except json.JSONDecodeError as e:
    raise ValueError("Invalid discovered_files_json") from e

if not isinstance(discovered_files, list):
    raise ValueError("Expected a JSON array of discovered files")

# Prepare CSV file metadata
csv_files = []
seen_paths = set()

for item in discovered_files:

    if not isinstance(item, dict):
        raise ValueError("Each discovered item must be an object")

    file_name = item.get("name", "")
    item_type = item.get("type", "")

    # Ignore folders and non-CSV files
    if item_type != "File":
        continue

    if not isinstance(file_name, str):
        raise ValueError("Invalid file name")

    if not file_name.lower().endswith(".csv"):
        continue

    # Prevent path traversal and unexpected filenames
    if (
        file_name in ("", ".", "..")
        or "/" in file_name
        or "\\" in file_name
    ):
        raise ValueError(f"Unsafe file name: {file_name}")

    source_path = (
        f"abfss://{source_container}@{storage_account}"
        f".dfs.core.windows.net/{source_folder}/{file_name}"
    )

    if source_path in seen_paths:
        continue

    seen_paths.add(source_path)

    csv_files.append({
        "file_name": file_name,
        "source_file": source_path
    })

print("Discovered items:", len(discovered_files))
print("Eligible CSV files:", len(csv_files))

for file in csv_files:
    print(file["source_file"])


In [0]:

# ============================================================
# Cell 3 - Identify unregistered CSV files
# ============================================================

from pyspark.sql import functions as F

# Read previously registered source paths
registered_rows = (
    spark.table(control_table)
    .select("source_file")
    .distinct()
    .collect()
)

registered_paths = {
    row["source_file"]
    for row in registered_rows
    if row["source_file"] is not None
}

# Identify new files only
new_files = [
    file
    for file in csv_files
    if file["source_file"] not in registered_paths
]

print("CSV files discovered:", len(csv_files))
print("New files requiring registration:", len(new_files))
print("Already registered:", len(csv_files) - len(new_files))

for file in new_files:
    print("NEW:", file["source_file"])


In [0]:

# ============================================================
# Cell 4 - Register new CSV files
# ============================================================

import uuid
from datetime import datetime, timezone

# Prepare registration records
registration_records = []

for file in new_files:
    registration_records.append({
        "file_id": str(uuid.uuid4()),
        "batch_id": f"BATCH_{uuid.uuid4().hex[:12].upper()}",
        "source_container": source_container,
        "source_folder": source_folder,
        "file_name": file["file_name"],
        "source_file": file["source_file"],
        "file_checksum": None,
        "load_type": "INCREMENTAL",
        "is_enabled": True,
        "processing_stage": "REGISTERED",
        "pipeline_status": "PENDING",
        "retry_count": 0,
        "adf_run_id": None,
        "registered_at": datetime.now(timezone.utc),
        "processing_start_ts": None,
        "processing_end_ts": None,
        "last_updated_ts": datetime.now(timezone.utc),
        "error_message": None
    })

print("Files prepared for registration:", len(registration_records))

# Do not write until duplicate-safe registration is implemented.


In [0]:

# ============================================================
# Cell 5 - Register new files using Delta MERGE
# ============================================================

from pyspark.sql.types import (
    StructType, StructField, StringType,
    BooleanType, IntegerType, TimestampType
)
from delta.tables import DeltaTable

registration_schema = StructType([
    StructField("file_id", StringType()),
    StructField("batch_id", StringType()),
    StructField("source_container", StringType()),
    StructField("source_folder", StringType()),
    StructField("file_name", StringType()),
    StructField("source_file", StringType()),
    StructField("file_checksum", StringType()),
    StructField("load_type", StringType()),
    StructField("is_enabled", BooleanType()),
    StructField("processing_stage", StringType()),
    StructField("pipeline_status", StringType()),
    StructField("retry_count", IntegerType()),
    StructField("adf_run_id", StringType()),
    StructField("registered_at", TimestampType()),
    StructField("processing_start_ts", TimestampType()),
    StructField("processing_end_ts", TimestampType()),
    StructField("last_updated_ts", TimestampType()),
    StructField("error_message", StringType())
])

if registration_records:
    df_registration = spark.createDataFrame(
        registration_records,
        schema=registration_schema
    )

    target = DeltaTable.forName(
        spark,
        "dbw_healthcare_de_dev.etl.file_control"
    )

    (
        target.alias("target")
        .merge(
            df_registration.alias("source"),
            "target.source_file = source.source_file"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("Registration MERGE completed")

else:
    print("No new files to register")


In [0]:

# ============================================================
# Cell 6 - Verify file registration
# ============================================================

from pyspark.sql import functions as F

source_paths = [f["source_file"] for f in csv_files]

if source_paths:

    df_registered = (
        spark.table(control_table)
        .filter(F.col("source_file").isin(source_paths))
        .select(
            "file_id",
            "batch_id",
            "file_name",
            "source_file",
            "pipeline_status",
            "processing_stage",
            "is_enabled",
            "registered_at"
        )
    )

    display(df_registered.orderBy("file_name"))

    # Verify that each discovered CSV has exactly one registration
    registration_counts = (
        df_registered
        .groupBy("source_file")
        .count()
        .collect()
    )

    count_by_path = {
        row["source_file"]: row["count"]
        for row in registration_counts
    }

    invalid_paths = [
        path for path in source_paths
        if count_by_path.get(path, 0) != 1
    ]

    if invalid_paths:
        raise RuntimeError(
            "Registration verification failed. "
            f"Missing or duplicate registrations: {invalid_paths}"
        )

    print("Registration verification passed")
    print("Discovered CSV files:", len(source_paths))

else:
    print("No CSV files discovered. Nothing to verify.")
